# Causal LM 미세조정 (GPT 계열) — Disaster Tweets (PyTorch + transformers) — Colab

사전학습된 **생성형 언어모델 `distilgpt2`** 를 트윗 코퍼스에 **미세조정(fine-tuning)** 해 텍스트를 생성하는 기본 예제입니다.

- 핵심 흐름: 코퍼스 토크나이즈 → **Causal LM(다음 토큰 예측) 학습** → `generate()` 로 새 문장 생성.
- IOAI 2024 NLP 과제("언어모델을 코퍼스에 미세조정")의 기반 기법입니다. (우리 12번 BERT는 *분류*, 이건 *생성형* LM)
- [NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started) 의 트윗 텍스트를 코퍼스로 사용합니다.
- 토큰이 **노트북에 내장**되어 바로 실행됩니다. 생성 데모라 제출 리더보드는 없습니다.

> ⚙️ **GPU 권장**: 런타임 → 런타임 유형 변경 → GPU.  
> ⚠️ 노트북에 개인 API 토큰이 평문으로 들어 있습니다. 외부에 공유/업로드하지 마세요.

## 0. Setup — 라이브러리 설치 & Kaggle 자격증명

In [ ]:
import sys, subprocess
for pkg in ["kaggle", "torch", "transformers", "numpy", "pandas"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=False)
print("라이브러리 준비 완료")

In [ ]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_5dd261fded8e0d7eb2c29d8d65dfabea"
print("Kaggle 자격증명 설정 완료 (내장 토큰)")

## 1. Kaggle에서 트윗 데이터 다운로드

In [ ]:
import os, glob, zipfile
WORK_DIR = "/content" if os.path.isdir("/content") else os.getcwd()
from kaggle.api.kaggle_api_extended import KaggleApi
api = KaggleApi(); api.authenticate()
api.competition_download_files("nlp-getting-started", path=WORK_DIR, quiet=False)
for zp in glob.glob(os.path.join(WORK_DIR, "*.zip")):
    with zipfile.ZipFile(zp) as zf: zf.extractall(WORK_DIR)
    os.remove(zp)
print("다운로드 완료:", [f for f in sorted(os.listdir(WORK_DIR)) if f.endswith(".csv")])

## 2. 코퍼스 준비 & 토크나이즈

트윗 텍스트를 이어붙여 토큰화한 뒤, 고정 길이 블록으로 잘라 학습 샘플을 만듭니다.

In [ ]:
import pandas as pd, torch
from transformers import AutoTokenizer

train = pd.read_csv(os.path.join(WORK_DIR, "train.csv"))
texts = train["text"].fillna("").tolist()

MODEL_NAME = "distilgpt2"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

# 모든 트윗을 EOS 로 이어붙여 하나의 토큰 스트림 생성
eos = tokenizer.eos_token
big_text = (" " + eos + " ").join(texts)
ids = tokenizer(big_text, return_tensors="pt")["input_ids"][0]

BLOCK = 64
n_blocks = ids.size(0) // BLOCK
blocks = ids[: n_blocks * BLOCK].view(n_blocks, BLOCK)
print("tokens:", ids.size(0), "| blocks:", tuple(blocks.shape))

## 3. DataLoader & 모델 로드 (`distilgpt2`)

In [ ]:
from torch.utils.data import TensorDataset, DataLoader
from transformers import AutoModelForCausalLM

loader = DataLoader(TensorDataset(blocks), batch_size=16, shuffle=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(device)
print("device:", device, "| 모델 로드 완료:", MODEL_NAME)

## 4. 미세조정 학습 (다음 토큰 예측)

Causal LM 은 `labels=input_ids` 로 주면 내부에서 한 칸 시프트해 다음 토큰 손실을 계산합니다.

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)
EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    model.train(); running = 0.0; n = 0
    for (batch,) in loader:
        batch = batch.to(device)
        out = model(input_ids=batch, labels=batch)
        out.loss.backward(); optimizer.step(); optimizer.zero_grad()
        running += out.loss.item() * batch.size(0); n += batch.size(0)
    ppl = torch.exp(torch.tensor(running / n))
    print(f"epoch {epoch} | LM loss {running/n:.3f} | perplexity {ppl:.1f}")

## 5. 텍스트 생성

프롬프트를 주고 미세조정된 모델로 이어쓰기를 합니다.

In [ ]:
model.eval()
prompts = ["Breaking news:", "The fire", "I just saw"]
for p in prompts:
    ids_in = tokenizer(p, return_tensors="pt").input_ids.to(device)
    out = model.generate(ids_in, max_new_tokens=40, do_sample=True, top_k=50,
                         top_p=0.95, temperature=0.9, pad_token_id=tokenizer.eos_token_id)
    print("•", tokenizer.decode(out[0], skip_special_tokens=True).replace("\n", " "))
    print()

## 🏁 제출 안내

이 노트북은 **텍스트 생성(Causal LM 미세조정)** 데모라 제출 리더보드가 없습니다.

- 코퍼스 출처: **[NLP Getting Started](https://www.kaggle.com/c/nlp-getting-started)**

> IOAI 2024 NLP 과제(언어모델을 새 코퍼스에 미세조정)의 기반 기법입니다. 같은 대회의 12번(BERT)은 *분류*, 본 노트북은 *생성형* 언어모델 미세조정입니다.